In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = "/content/drive/MyDrive/RoofMaterialProject"
MANIFEST = f"{ROOT}/patches/train/patch_manifest.csv"

df = pd.read_csv(MANIFEST)

# Select only the 23 buildings labelled "other"
other = df[
    (df["roof_material"] == "other") &
    (df["status"] == "success")
].copy()

for region in ["borde_rural", "borde_soacha"]:
    subset = other[other["region"] == region].reset_index(drop=True)

    print(f"\n{region}: {len(subset)} patches")

    n = len(subset)
    cols = 4
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(
        rows, cols, figsize=(12, 3.5 * rows)
    )
    axes = axes.flatten()

    for i, row in subset.iterrows():
        path = row["patch_path"]

        if os.path.exists(path):
            img = Image.open(path).convert("RGB")
            axes[i].imshow(img)
            axes[i].set_title(
                f"ID: {row['building_id']}", fontsize=9
            )
        else:
            axes[i].set_title(
                f"Image missing: {row['building_id']}"
            )

        axes[i].axis("off")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    plt.suptitle(
        f"{region} — Roofs labelled OTHER", fontsize=14
    )
    plt.tight_layout()
    plt.show()


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
!pip -q install scikit-image



In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from skimage.color import rgb2gray
from skimage.feature import (
    local_binary_pattern,
    graycomatrix,
    graycoprops
)

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
ROOT = Path("/content/drive/MyDrive/RoofMaterialProject")
MANIFEST = ROOT / "patches" / "train" / "patch_manifest.csv"

OUTPUT = ROOT / "features" / "roof_features.csv"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(MANIFEST)

print("Manifest rows:", len(df))
print(df["roof_material"].value_counts())

# --------------------------------------------------
# 2. Helper: resolve patch path
# --------------------------------------------------
def resolve_patch_path(path):
    p = Path(str(path))

    if p.exists():
        return p

    # Handle relative paths if present in the manifest
    candidate = ROOT / p
    if candidate.exists():
        return candidate

    raise FileNotFoundError(f"Patch not found: {path}")


# --------------------------------------------------
# 3. Feature extraction for one image
# --------------------------------------------------
def extract_features(image_path):
    with Image.open(image_path) as im:
        rgb = np.asarray(im.convert("RGB"), dtype=np.uint8)
        hsv = np.asarray(im.convert("HSV"), dtype=np.uint8)

    features = {}

    # A. RGB colour statistics
    for i, channel in enumerate(["r", "g", "b"]):
        values = rgb[:, :, i].astype(np.float32)

        features[f"{channel}_mean"] = values.mean()
        features[f"{channel}_std"] = values.std()
        features[f"{channel}_p10"] = np.percentile(values, 10)
        features[f"{channel}_p90"] = np.percentile(values, 90)

    # B. HSV colour statistics
    for i, channel in enumerate(["h", "s", "v"]):
        values = hsv[:, :, i].astype(np.float32)

        features[f"{channel}_mean"] = values.mean()
        features[f"{channel}_std"] = values.std()

    # C. Convert RGB to grayscale for texture analysis
    gray = rgb2gray(rgb)
    gray_uint8 = np.round(gray * 255).astype(np.uint8)

    # D. LBP: 8 neighbours, radius 1, uniform patterns
    P = 8
    R = 1

    lbp = local_binary_pattern(
        gray_uint8, P, R, method="uniform"
    )

    # Uniform LBP with P=8 has P+2 histogram bins
    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, P + 4),
        range=(0, P + 2)
    )

    lbp_hist = lbp_hist.astype(np.float64)
    lbp_hist /= max(lbp_hist.sum(), 1)

    for i, value in enumerate(lbp_hist):
        features[f"lbp_bin_{i}"] = value

    # E. GLCM: reduce grayscale to 16 intensity levels
    quantized = (gray_uint8 // 16).astype(np.uint8)

    distances = [1, 2]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]

    glcm = graycomatrix(
        quantized,
        distances=distances,
        angles=angles,
        levels=16,
        symmetric=True,
        normed=True
    )

    # Average and standard deviation across directions
    properties = [
        "contrast",
        "homogeneity",
        "energy",
        "correlation"
    ]

    for prop in properties:
        values = graycoprops(glcm, prop)

        for j, distance in enumerate(distances):
            features[f"glcm_{prop}_d{distance}_mean"] = (
                np.nanmean(values[j, :])
            )
            features[f"glcm_{prop}_d{distance}_std"] = (
                np.nanstd(values[j, :])
            )

    # Replace any undefined numerical values
    for key, value in features.items():
        if not np.isfinite(value):
            features[key] = 0.0

    return features


# --------------------------------------------------
# 4. Process all patches
# --------------------------------------------------
records = []
errors = []

for row in tqdm(df.itertuples(index=False), total=len(df)):
    try:
        patch = resolve_patch_path(row.patch_path)
        feature_values = extract_features(patch)

        records.append({
            "region": row.region,
            "building_id": row.building_id,
            **feature_values
        })

    except Exception as e:
        errors.append({
            "region": row.region,
            "building_id": row.building_id,
            "patch_path": row.patch_path,
            "error": str(e)
        })

# --------------------------------------------------
# 5. Save features and error report
# --------------------------------------------------
features_df = pd.DataFrame(records)

features_df.to_csv(OUTPUT, index=False)

print("\nFeature extraction completed.")
print("Feature rows:", len(features_df))
print("Feature columns:", len(features_df.columns))
print("Failed patches:", len(errors))
print("Saved to:", OUTPUT)

if errors:
    error_path = ROOT / "features" / "feature_extraction_errors.csv"
    pd.DataFrame(errors).to_csv(error_path, index=False)
    print("Error report:", error_path)

print("\nFirst five feature rows:")
display(features_df.head())


Manifest rows: 10356
roof_material
healthy_metal      5544
irregular_metal    3632
incomplete          660
concrete_cement     497
other                23
Name: count, dtype: int64


  0%|          | 0/10356 [00:00<?, ?it/s]


Feature extraction completed.
Feature rows: 10356
Feature columns: 47
Failed patches: 0
Saved to: /content/drive/MyDrive/RoofMaterialProject/features/roof_features.csv

First five feature rows:


,region,building_id,r_mean,r_std,r_p10,r_p90,g_mean,g_std,g_p10,g_p90,...,glcm_homogeneity_d2_mean,glcm_homogeneity_d2_std,glcm_energy_d1_mean,glcm_energy_d1_std,glcm_energy_d2_mean,glcm_energy_d2_std,glcm_correlation_d1_mean,glcm_correlation_d1_std,glcm_correlation_d2_mean,glcm_correlation_d2_std
0,borde_rural,7a3f2a10,154.192688,41.224415,104.0,215.0,152.169189,39.443832,114.0,219.0,...,0.716576,0.023135,0.245612,0.009911,0.228990,0.007738,0.924221,0.018739,0.885576,0.023539
1,borde_rural,7a1f731e,169.538818,52.308949,99.0,226.0,173.216003,52.750259,108.0,232.0,...,0.726747,0.020759,0.297484,0.007275,0.283835,0.006744,0.950274,0.012106,0.922521,0.016573
2,borde_rural,7a424ad8,176.453125,56.983269,92.0,227.0,182.863770,54.658607,104.0,232.0,...,0.754134,0.019627,0.352374,0.009771,0.337074,0.008643,0.961472,0.009790,0.938960,0.013343
3,borde_rural,7a3edc5e,203.982239,25.845503,171.0,236.0,204.990601,32.278107,162.0,246.0,...,0.881486,0.016003,0.367016,0.007124,0.353933,0.006974,0.976505,0.004994,0.965635,0.006880
4,borde_rural,7a303a6e,178.080750,48.833603,101.0,225.0,181.016235,50.835526,102.0,232.0,...,0.752217,0.026808,0.274406,0.009141,0.258877,0.008574,0.955883,0.011462,0.930733,0.016594


In [ ]:
"""changing the features from 45 to 44 as i bin is extra"""
import pandas as pd

path = "/content/drive/MyDrive/RoofMaterialProject/features/roof_features.csv"

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Feature columns:", len([
    c for c in df.columns
    if c not in ["region", "building_id"]
]))
print(df.columns.tolist())

Rows: 10356
Columns: 47
Feature columns: 45
['region', 'building_id', 'r_mean', 'r_std', 'r_p10', 'r_p90', 'g_mean', 'g_std', 'g_p10', 'g_p90', 'b_mean', 'b_std', 'b_p10', 'b_p90', 'h_mean', 'h_std', 's_mean', 's_std', 'v_mean', 'v_std', 'lbp_bin_0', 'lbp_bin_1', 'lbp_bin_2', 'lbp_bin_3', 'lbp_bin_4', 'lbp_bin_5', 'lbp_bin_6', 'lbp_bin_7', 'lbp_bin_8', 'lbp_bin_9', 'lbp_bin_10', 'glcm_contrast_d1_mean', 'glcm_contrast_d1_std', 'glcm_contrast_d2_mean', 'glcm_contrast_d2_std', 'glcm_homogeneity_d1_mean', 'glcm_homogeneity_d1_std', 'glcm_homogeneity_d2_mean', 'glcm_homogeneity_d2_std', 'glcm_energy_d1_mean', 'glcm_energy_d1_std', 'glcm_energy_d2_mean', 'glcm_energy_d2_std', 'glcm_correlation_d1_mean', 'glcm_correlation_d1_std', 'glcm_correlation_d2_mean', 'glcm_correlation_d2_std']


In [2]:
import pandas as pd
from pathlib import Path

source = "/content/drive/MyDrive/RoofMaterialProject/features/roof_features.csv"
output = "/content/drive/MyDrive/RoofMaterialProject/features/roof_features_44.csv"

df = pd.read_csv(source)

# Remove only the extra LBP histogram column
df_44 = df.drop(columns=["lbp_bin_10"])

# Validate
feature_cols = [
    c for c in df_44.columns
    if c not in ["region", "building_id"]
]

assert len(df_44) == 10356, "Unexpected row count"
assert len(feature_cols) == 44, f"Expected 44 features, got {len(feature_cols)}"
assert not df_44.duplicated(["region", "building_id"]).any(), "Duplicate building IDs"

df_44.to_csv(output, index=False)

print("Rows:", len(df_44))
print("Total columns:", len(df_44.columns))
print("Numerical feature columns:", len(feature_cols))
print("Saved to:", output)

Rows: 10356
Total columns: 46
Numerical feature columns: 44
Saved to: /content/drive/MyDrive/RoofMaterialProject/features/roof_features_44.csv


In [4]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/RoofMaterialProject/features/roof_features.csv"
)

print("Unique values in lbp_bin_10:",
      df["lbp_bin_10"].unique())

print("Nonzero rows:",
      (df["lbp_bin_10"] != 0).sum())

Unique values in lbp_bin_10: [0.]
Nonzero rows: 0


In [6]:
"""joining of patch_manifest.csv with the roff_features_44.csv"""

'joining of patch_manifest.csv with the roff_features_44.csv'

In [7]:

from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
ROOT = Path("/content/drive/MyDrive/RoofMaterialProject")

FEATURES = ROOT / "features" / "roof_features_44.csv"
MANIFEST = ROOT / "patches" / "train" / "patch_manifest.csv"

OUTPUT = ROOT / "features" / "roof_features_44_with_labels.csv"

# --------------------------------------------------
# 2. Load files
# --------------------------------------------------
features = pd.read_csv(FEATURES)
manifest = pd.read_csv(MANIFEST)

KEYS = ["region", "building_id"]

print("Feature rows:", len(features))
print("Manifest rows:", len(manifest))

# --------------------------------------------------
# 3. Validate required columns
# --------------------------------------------------
assert all(c in features.columns for c in KEYS), \
    "Missing join keys in feature file"

assert all(c in manifest.columns for c in KEYS), \
    "Missing join keys in manifest"

assert "roof_material" in manifest.columns, \
    "roof_material missing from manifest"

# --------------------------------------------------
# 4. Check duplicate building keys
# --------------------------------------------------
assert not features.duplicated(KEYS).any(), \
    "Duplicate building keys in features"

assert not manifest.duplicated(KEYS).any(), \
    "Duplicate building keys in manifest"

# --------------------------------------------------
# 5. Identify numerical features
# --------------------------------------------------
feature_cols = [
    c for c in features.columns
    if c not in KEYS
]

assert len(feature_cols) == 44, \
    f"Expected 44 features, found {len(feature_cols)}"

# Check that feature values are numeric and finite
features[feature_cols] = features[feature_cols].apply(
    pd.to_numeric, errors="raise"
)

assert np.isfinite(
    features[feature_cols].to_numpy(dtype=float)
).all(), "Features contain NaN or infinite values"

# --------------------------------------------------
# 6. Join features to manifest
# --------------------------------------------------
combined = manifest.merge(
    features,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True
)

# Every manifest building must have matching features
unmatched = combined["_merge"] != "both"

assert not unmatched.any(), (
    f"{unmatched.sum()} manifest buildings have no matching features"
)

# Also ensure no feature-only buildings were omitted
feature_keys = features[KEYS].drop_duplicates()
manifest_keys = manifest[KEYS].drop_duplicates()

key_check = feature_keys.merge(
    manifest_keys,
    on=KEYS,
    how="left",
    indicator=True
)

assert (key_check["_merge"] == "both").all(), \
    "Some feature rows have no matching manifest building"

combined = combined.drop(columns="_merge")

# --------------------------------------------------
# 7. Final validation and save
# --------------------------------------------------
assert len(combined) == len(manifest)
assert combined["roof_material"].notna().all()
assert len(feature_cols) == 44

combined.to_csv(OUTPUT, index=False)

print("\nJOIN SUCCESSFUL")
print("Rows:", len(combined))
print("Total columns:", len(combined.columns))
print("Numerical features:", len(feature_cols))
print("\nRoof-material counts:")
print(combined["roof_material"].value_counts())
print("\nSaved to:", OUTPUT)

display(combined.head())


Feature rows: 10356
Manifest rows: 10356

JOIN SUCCESSFUL
Rows: 10356
Total columns: 50
Numerical features: 44

Roof-material counts:
roof_material
healthy_metal      5544
irregular_metal    3632
incomplete          660
concrete_cement     497
other                23
Name: count, dtype: int64

Saved to: /content/drive/MyDrive/RoofMaterialProject/features/roof_features_44_with_labels.csv


,region,building_id,roof_material,patch_path,status,error,r_mean,r_std,r_p10,r_p90,...,glcm_homogeneity_d2_mean,glcm_homogeneity_d2_std,glcm_energy_d1_mean,glcm_energy_d1_std,glcm_energy_d2_mean,glcm_energy_d2_std,glcm_correlation_d1_mean,glcm_correlation_d1_std,glcm_correlation_d2_mean,glcm_correlation_d2_std
0,borde_rural,7a3f2a10,concrete_cement,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,154.19269,41.224415,104.0,215.0,...,0.716576,0.023135,0.245612,0.009911,0.228990,0.007738,0.924221,0.018739,0.885576,0.023539
1,borde_rural,7a1f731e,irregular_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,169.53882,52.308950,99.0,226.0,...,0.726747,0.020759,0.297484,0.007275,0.283835,0.006744,0.950274,0.012106,0.922521,0.016573
2,borde_rural,7a424ad8,healthy_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,176.45312,56.983270,92.0,227.0,...,0.754134,0.019627,0.352374,0.009771,0.337074,0.008643,0.961472,0.009790,0.938960,0.013343
3,borde_rural,7a3edc5e,healthy_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,203.98224,25.845503,171.0,236.0,...,0.881486,0.016003,0.367016,0.007124,0.353933,0.006974,0.976505,0.004994,0.965635,0.006880
4,borde_rural,7a303a6e,healthy_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,178.08075,48.833603,101.0,225.0,...,0.752217,0.026808,0.274406,0.009141,0.258877,0.008574,0.955883,0.011462,0.930733,0.016594


In [8]:

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/RoofMaterialProject")

DATA = ROOT / "features" / "roof_features_44_with_labels.csv"
df = pd.read_csv(DATA)

print("Dataset shape:", df.shape)
print("\nRegions:")
print(df["region"].value_counts())

print("\nRoof-material counts by region:")
print(pd.crosstab(df["region"], df["roof_material"], margins=True))

# Four-class dataset size
four_class = df[df["roof_material"] != "other"].copy()

print("\nFour-class dataset rows:", len(four_class))
print("\nFour-class counts by region:")
print(pd.crosstab(
    four_class["region"],
    four_class["roof_material"],
    margins=True
))


Dataset shape: (10356, 50)

Regions:
region
borde_soacha    5995
borde_rural     4361
Name: count, dtype: int64

Roof-material counts by region:
roof_material  concrete_cement  healthy_metal  incomplete  irregular_metal  \
region                                                                       
borde_rural                217           2383         245             1508   
borde_soacha               280           3161         415             2124   
All                        497           5544         660             3632   

roof_material  other    All  
region                       
borde_rural        8   4361  
borde_soacha      15   5995  
All               23  10356  

Four-class dataset rows: 10333

Four-class counts by region:
roof_material  concrete_cement  healthy_metal  incomplete  irregular_metal  \
region                                                                       
borde_rural                217           2383         245             1508   
borde_soacha      

In [9]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/RoofMaterialProject")
MANIFEST = ROOT / "patches" / "train" / "patch_manifest.csv"

manifest = pd.read_csv(MANIFEST)

print("Manifest columns:")
print(manifest.columns.tolist())

print("\nFirst five rows:")
display(manifest.head())

spatial_keywords = [
    "latitude", "longitude", "lat", "lon",
    "x", "y", "geometry", "wkt",
    "centroid", "easting", "northing"
]

print("\nPotential spatial columns:")
print([
    col for col in manifest.columns
    if any(word in col.lower() for word in spatial_keywords)
])


Manifest columns:
['region', 'building_id', 'roof_material', 'patch_path', 'status', 'error']

First five rows:


,region,building_id,roof_material,patch_path,status,error
0,borde_rural,7a3f2a10,concrete_cement,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN
1,borde_rural,7a1f731e,irregular_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN
2,borde_rural,7a424ad8,healthy_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN
3,borde_rural,7a3edc5e,healthy_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN
4,borde_rural,7a303a6e,healthy_metal,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN



Potential spatial columns:
[]


In [11]:
"""extracting 4classes only"""

'extracting 4classes only'

In [12]:

import pandas as pd

ROOT = "/content/drive/MyDrive/RoofMaterialProject"
INPUT = f"{ROOT}/features/roof_features_44_with_labels.csv"
OUTPUT = f"{ROOT}/features/roof_features_4class.csv"

df = pd.read_csv(INPUT)

# Exclude "other" from this experiment; preserve the original file
data = df[df["roof_material"] != "other"].copy()

# Verify the dataset
print("Rows:", len(data))
print("Features:", sum(
    c.startswith(("r_", "g_", "b_", "h_", "s_", "v_", "lbp_", "glcm_"))
    for c in data.columns
))
print("\nClass counts:")
print(data["roof_material"].value_counts())

# Save as a separate file
data.to_csv(OUTPUT, index=False)

print("\nSaved:", OUTPUT)


Rows: 10333
Features: 44

Class counts:
roof_material
healthy_metal      5544
irregular_metal    3632
incomplete          660
concrete_cement     497
Name: count, dtype: int64

Saved: /content/drive/MyDrive/RoofMaterialProject/features/roof_features_4class.csv
